# SST factorization — QC

Is each fit trustworthy, and how do the fits differ? Interpretation of what the factors
*mean* lives in `02_explore.ipynb`; this notebook only asks whether the models are sound
and comparable.

Reads, for each fit, the two artifacts the runners write:

- `/scratch/sst-drvi/models/<run>/<model>/model.pt` — design, registry and the full
  per-epoch loss history. Read on CPU without constructing the model.
- `/scratch/sst-drvi/embeddings/<run>_<model>_embed.h5ad` — latent space, per-dimension
  stats and the latent UMAP.

CPU-only; everything GPU-bound already happened in `train_drvi.py` / `train_scvi.py`.
Kernel: `sst-drvi`.

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

import drvi
import sst_drvi as S

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(4, 4))
pd.set_option("display.max_rows", 200)

## Helpers used more than once

`load_history` and `factor_association` run once per fit, `covariate_design` twice, and
`model_design` from three places, so they are named here. Anything this notebook calls
once is written out where it is used.

In [ ]:
# Each of these has several callers below -- `load_history` and `factor_association` run
# once per fit, `covariate_design` twice, `model_design` from three places. Anything
# called once is written out where it is used instead. `logger` is wired up because these
# readers report what they skip: missing obs columns, collapsed latent dimensions.
import logging
from collections import Counter
from collections.abc import Sequence
from pathlib import Path

logger = logging.getLogger("qc")
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())
    logger.setLevel(logging.INFO)


def as_list(value) -> list:
    """Registry entries are sometimes numpy arrays, whose truthiness raises."""
    return [] if value is None else list(value)


def model_file(model_path: Path | str) -> Path:
    path = Path(model_path)
    return path / "model.pt" if path.is_dir() else path


def load_saved(model_path: Path | str) -> dict:
    """``torch.load`` a saved scvi-tools model without constructing the model."""
    import torch

    path = model_file(model_path)
    if not path.exists():
        raise FileNotFoundError(f"{path} missing; train the model first")
    return torch.load(path, map_location="cpu", weights_only=False)


def load_history(model_path: Path | str) -> pd.DataFrame:
    """Epochs x metrics training history, read out of a saved model.

    scvi-tools keeps every logged loss component in ``attr_dict["history_"]`` -- ELBO,
    reconstruction loss, the KL terms and the warmup weight, train and validation each --
    so the loss curves need no retraining and no reload of the module.

    Most metrics are logged once per epoch, but not all: with ``reduce_lr_on_plateau`` the
    learning rate is logged once per optimizer STEP. Concatenating that on an outer join
    unions a 0..599 epoch axis with a 0..122196 step axis, so every epoch metric becomes
    NaN on the step rows and ``.iloc[-1]`` quietly returns NaN rather than the final
    epoch's value. Step-indexed series are therefore binned onto the epoch axis (last
    value within each epoch) instead of being allowed to widen it.
    """
    history = load_saved(model_path)["attr_dict"].get("history_") or {}
    if not history:
        raise ValueError(f"{model_file(model_path)} has no training history")

    n_epochs = Counter(len(v) for v in history.values()).most_common(1)[0][0]
    per_epoch, rebinned = {}, []
    for name, frame in history.items():
        if len(frame) == n_epochs:
            per_epoch[name] = frame.reset_index(drop=True)
            continue
        edges = np.linspace(0, len(frame), n_epochs + 1).astype(int)
        values = [frame.iloc[lo:hi].iloc[-1] for lo, hi in zip(edges[:-1], edges[1:]) if hi > lo]
        if len(values) != n_epochs:  # too few samples to bin; not worth guessing
            logger.info("load_history: dropping %r (%d rows, %d epochs)", name, len(frame), n_epochs)
            continue
        per_epoch[name] = pd.DataFrame(values).reset_index(drop=True)
        rebinned.append(name)
    if rebinned:
        logger.info("load_history: binned %s onto the epoch axis", ", ".join(sorted(rebinned)))

    out = pd.concat(per_epoch.values(), axis=1)
    out.index = pd.RangeIndex(n_epochs, name="epoch")
    return out


def model_design(model_path: Path | str) -> dict:
    """What a saved model is: constructor arguments, registry, size and how far it trained."""
    saved = load_saved(model_path)
    attrs = saved["attr_dict"]
    registry = attrs["registry_"]

    init = dict(attrs["init_params_"].get("non_kwargs") or {})
    # both SCVI and DRVI funnel the module-level knobs through a nested "kwargs" entry
    init.update((attrs["init_params_"].get("kwargs") or {}).get("kwargs") or {})

    stats: dict = {}
    for field in registry["field_registries"].values():
        stats.update(field.get("summary_stats") or {})

    n_params: dict[str, int] = {}
    for key, tensor in saved["model_state_dict"].items():
        if hasattr(tensor, "numel"):
            head = key.split(".")[0]
            n_params[head] = n_params.get(head, 0) + tensor.numel()

    history = attrs.get("history_") or {}
    return {
        "model": registry.get("model_name"),
        "scvi_version": registry.get("scvi_version"),
        "setup_args": dict(registry.get("setup_args") or {}),
        "field_registries": registry["field_registries"],
        "init_params": init,
        "summary_stats": stats,
        "n_params": n_params,
        "n_params_total": sum(n_params.values()),
        "epochs_run": len(history.get("elbo_train", ())),
        "is_trained": bool(attrs.get("is_trained_", False)),
        # these are numpy arrays, so `or ()` would raise on the ambiguous truth value
        "n_train": 0 if attrs.get("train_indices_") is None else len(attrs["train_indices_"]),
        "n_validation": (
            0
            if attrs.get("validation_indices_") is None
            else len(attrs["validation_indices_"])
        ),
    }


def covariate_design(
    model_path: Path | str,
    tested: Sequence[str],
    scaling: dict | None = None,
) -> pd.DataFrame:
    """How each covariate in ``tested`` enters the model -- including the ones that do not.

    The ``not modelled`` rows are the point of this table. A covariate the model was never
    told about, which then explains much of a latent dimension, means a dimension was
    spent on nuisance structure; a covariate marked ``batch key`` that still scores high
    means structure leaked past the correction. Read it beside `factor_association`.

    ``tested`` should be the same covariate list the association test uses, so nothing
    that gets scored is missing from the table. Pass ``scaling`` --
    ``embed.uns["covariate_scaling"]``, written by `scale_continuous_covariates` --
    to report whether a continuous covariate was log1p'd as well as z-scored; without it
    the transform is reported as the generic "standardized".
    """
    scaling = scaling or {}
    d = model_design(model_path)
    setup, stats, fields = d["setup_args"], d["summary_stats"], d["field_registries"]
    init = d["init_params"]

    batch_key = setup.get("batch_key")
    cat_state = (fields.get("extra_categorical_covs") or {}).get("state_registry") or {}
    cat_levels = dict(
        zip(
            as_list(cat_state.get("field_keys")),
            as_list(cat_state.get("n_cats_per_key")),
            strict=False,
        )
    )
    cont_state = (fields.get("extra_continuous_covs") or {}).get("state_registry") or {}
    # continuous covariates are registered on the derived, standardized columns
    cont_keys = {
        str(col).removesuffix(S.SCALED_SUFFIX): str(col).endswith(S.SCALED_SUFFIX)
        for col in as_list(cont_state.get("columns"))
    }
    encoded = bool(init.get("encode_covariates"))

    rows = []
    for name in tested:
        if name == batch_key:
            role, representation = "batch key", init.get("batch_representation", "one-hot")
            levels = stats.get("n_batch")
        elif name in cat_levels:
            role, representation = "categorical covariate", "one-hot"
            levels = cat_levels[name]
        elif name in cont_keys:
            role = "continuous covariate"
            if not cont_keys[name]:
                representation = "as given"
            elif name in scaling:
                representation = (
                    "log1p + z-scored" if scaling[name].get("log1p") else "z-scored"
                )
            else:
                representation = "standardized"
            levels = None
        else:
            role, representation, levels = "not modelled", "-", None
        modelled = role != "not modelled"
        rows.append(
            {
                "role": role,
                "representation": representation,
                "n_levels": levels,
                "reaches_encoder": encoded if modelled else False,
                "reaches_decoder": modelled,
                "affects_dispersion": role == "batch key"
                and init.get("dispersion") == "gene-batch",
            }
        )
    return pd.DataFrame(rows, index=pd.Index(list(tested), name="covariate"))


def eta_squared_many(x: np.ndarray, codes: np.ndarray, n_groups: int) -> np.ndarray:
    """Eta-squared of every column of ``x`` against one integer-coded grouping.

    Vectorized with ``bincount``: a per-group Python loop is O(n x n_groups), which is
    unusable at 902 ``library_prep`` levels.
    """
    counts = np.bincount(codes, minlength=n_groups).astype(np.float64)
    live = counts > 0
    grand = x.mean(axis=0)
    total = ((x - grand) ** 2).sum(axis=0)

    out = np.full(x.shape[1], np.nan)
    if live.sum() < 2:
        return out
    for i in range(x.shape[1]):
        if total[i] == 0:
            continue
        sums = np.bincount(codes, weights=x[:, i], minlength=n_groups)
        means = sums[live] / counts[live]
        out[i] = (counts[live] * (means - grand[i]) ** 2).sum() / total[i]
    return out


def associate(
    x: np.ndarray,
    names,
    obs: pd.DataFrame,
    categorical: tuple[str, ...],
    continuous: tuple[str, ...],
) -> pd.DataFrame:
    """Association of every column of ``x`` with each named covariate.

    Categorical -> eta-squared (variance explained); continuous -> ``|Spearman rho|``.
    """
    from scipy import stats

    x = np.asarray(x, dtype=np.float64)
    out = pd.DataFrame(index=pd.Index(names), dtype=float)

    for col in categorical:
        if col not in obs:
            logger.warning("skipping missing obs column %s", col)
            continue
        cat = pd.Categorical(obs[col])
        keep = cat.codes >= 0
        out[col] = eta_squared_many(
            x[keep], cat.codes[keep].astype(np.intp), len(cat.categories)
        )

    for col in continuous:
        if col not in obs:
            logger.warning("skipping missing obs column %s", col)
            continue
        values = pd.to_numeric(obs[col], errors="coerce").to_numpy(dtype=float)
        keep = np.isfinite(values)
        if keep.sum() < 3:
            out[col] = np.nan
            continue
        # ranks every column against the covariate in one call
        rho = stats.spearmanr(x[keep], values[keep]).statistic
        out[col] = np.abs(np.atleast_2d(rho)[:-1, -1]) if x.shape[1] > 1 else abs(rho)
    return out


def factor_association(
    embed,
    categorical: tuple[str, ...] = (),
    continuous: tuple[str, ...] = (),
) -> pd.DataFrame:
    """Factors x covariates association matrix.

    Categorical covariates get eta-squared (variance explained, 0-1); continuous ones
    get ``|Spearman rho|``, so a single heatmap separates factors driven by biology
    from those tracking technical covariates.

    Associates on the signed factor value, one row per factor. `02_explore.ipynb` carries
    its own copy that can also split each factor into its two directions first; this
    notebook only ever asks the signed question, so that branch is not here.

    Only non-vanished factors are returned.
    """
    if {"order", "vanished", "title"} <= set(embed.var.columns):
        used = embed.var.sort_values("order")
        used = used[~used["vanished"]]
        x = np.asarray(embed[:, used.index].X)
        names = used["title"].to_numpy()
    else:
        # An scVI embedding has no vanished/order bookkeeping to reorder by, but it does
        # carry `used` from latent_stats -- and collapsed dimensions MUST be dropped here.
        # Eta-squared is a variance ratio, so it is scale-invariant and cannot tell a live
        # dimension from one sitting at the prior: a dimension with std 0.008 that wiggles
        # slightly with library scores just as high as a real one, which fills the heatmap
        # with noise that reads as signal.
        keep = embed.var.index
        if "used" in embed.var:
            keep = embed.var.index[embed.var["used"].to_numpy().astype(bool)]
            dropped = embed.n_vars - len(keep)
            if dropped:
                logger.info(
                    "dropping %d of %d collapsed latent dimensions (var['used'] is False); "
                    "eta-squared is scale-invariant and would score them like live ones",
                    dropped,
                    embed.n_vars,
                )
        subset = embed[:, keep]
        x = np.asarray(subset.X)
        names = (
            subset.var["title"].to_numpy()
            if "title" in subset.var
            else subset.var_names.to_numpy()
        )
    return associate(x, names, embed.obs, categorical, continuous)

## Fits under review

`(label, run, model)`. Anything not yet on disk is skipped with a notice rather than
raising, so this notebook is readable while the runs are still going. `drvi 1gpu` is the
original single-GPU baseline, kept as a control on whether moving to 4-GPU DDP — which
quadruples the effective batch to 1024 and disables early stopping — changed the fit.

In [ ]:
# The `_long` runs are the converged refits: a third-of-budget KL warmup shared by both
# model families, and enough epochs for the validation curve to flatten. The 200-epoch
# runs above them are kept for comparison, not for conclusions -- see section 3.
FITS = [
    ("drvi", "sst_k64", "drvi"),
    ("drvi+covar", "sst_k64_covar", "drvi"),
    ("scvi+covar", "sst_k64_covar", "scvi"),
    ("drvi 1gpu", "sst_k64_1gpu", "drvi"),
    ("drvi long", "sst_k64_long", "drvi"),
    ("drvi+covar long", "sst_k64_covar_long", "drvi"),
    ("scvi+covar long", "sst_k64_covar_long", "scvi"),
]

models, embeds = {}, {}
for label, run, kind in FITS:
    model_path = S.MODELS_DIR / run / kind
    embed_path = S.EMBED_DIR / f"{run}_{kind}_embed.h5ad"
    if not (model_path / "model.pt").exists():
        print(f"{label:12s} not fitted yet -> {model_path}")
        continue
    models[label] = model_path
    embeds[label] = ad.read_h5ad(embed_path) if embed_path.exists() else None
    state = "model + embedding" if embeds[label] is not None else "model only (no embedding yet)"
    print(f"{label:12s} {state}")

DRVI_FITS = [l for l, e in embeds.items() if e is not None and models[l].name == "drvi"]


def fit_config(label):
    """The config dict the runner stored in uns, whichever model wrote it."""
    embed = embeds.get(label)
    if embed is None:
        return {}
    for key in ("drvi_config", "scvi_config"):
        if key in embed.uns:
            return json.loads(embed.uns[key])
    return {}

## 1. Model design

What each fit actually is. Rows that do not apply to a model come back as `None` —
`split_method` is DRVI's alone — so the table doubles as a record of where the two model
families differ. `devices` and the effective batch come from the config the runner stored
in `uns`, not from the saved module, because they are training-time rather than
architectural.

In [ ]:
# One column per fit. Rows that do not apply to a model come back as None --
# `split_method` is DRVI's alone -- so this doubles as a record of where the families
# differ, which is what makes the ELBO comparison in section 3 readable at all.
columns = {}
for label, path in models.items():
    design_of = model_design(path)
    init = design_of["init_params"]
    stats_of = design_of["summary_stats"]
    setup_args = design_of["setup_args"]
    columns[label] = {
        "model": design_of["model"],
        "scvi-tools": design_of["scvi_version"],
        "n_latent": init.get("n_latent"),
        "n_hidden": init.get("n_hidden"),
        "n_layers": init.get("n_layers"),
        "gene_likelihood": init.get("gene_likelihood"),
        "dispersion": init.get("dispersion"),
        "batch_representation": init.get("batch_representation"),
        "encode_covariates": init.get("encode_covariates"),
        "split_method": init.get("split_method"),
        "split_aggregation": init.get("split_aggregation"),
        "batch_key": setup_args.get("batch_key"),
        "n_batch": stats_of.get("n_batch"),
        "n_extra_categorical_covs": stats_of.get("n_extra_categorical_covs"),
        "n_extra_continuous_covs": stats_of.get("n_extra_continuous_covs"),
        "n_cells": design_of["n_train"] + design_of["n_validation"],
        "n_genes": stats_of.get("n_vars"),
        "epochs_run": design_of["epochs_run"],
        "parameters": f"{design_of['n_params_total']:,}",
    }
design = pd.DataFrame(columns)

extra = {}
for label in design.columns:
    cfg = fit_config(label)
    devices = cfg.get("devices", 1)
    extra[label] = {
        "devices": devices,
        "batch_size (per device)": cfg.get("batch_size"),
        "effective batch": (cfg.get("batch_size") or 0) * devices or None,
        "max_epochs": cfg.get("max_epochs"),
        "kl_warmup_epochs": cfg.get("kl_warmup_epochs", cfg.get("max_epochs")),
    }
design = pd.concat([design, pd.DataFrame(extra)])
design

In [ ]:
# The registry is the audit trail for what was actually registered, as opposed to what
# the command line asked for.
for label, path in models.items():
    print(f"--- {label} ---")
    print(json.dumps(model_design(path)["setup_args"], indent=2))

## 2. How are the covariates modelled?

One row per covariate this notebook later associates against, so nothing that gets scored
is missing here. `role` is the important column, and `not modelled` is the important
value: a covariate the model was never told about, which then explains much of a latent
dimension (section 6), means a dimension was spent on nuisance structure. A covariate
marked `batch key` that still scores high means structure leaked past the correction.

`reaches_encoder` is `False` throughout unless a fit was given `--encode-covariates`.
Decoder-only means the reconstruction can source a covariate-driven gene from the
covariate instead of from `z`, which removes the *pressure* on the latent space to encode
it — but a covariate-blind encoder can still put it there. So this table sets up the
question that section 6 answers; it does not settle it.

In [ ]:
CATEGORICAL = ("Supertype", "Brain Region", "Donor ID", "library_prep", "method",
               "Chemistry", "Sex", "Cognitive Status", "Overall AD neuropathological Change",
               "Braak")
CONTINUOUS = ("CPS_Local", "CPS_Global", "CPS_Local_pTau", "CPS_Local_ABeta",
              "Age at Death", "PMI", "Number of UMIs", "Genes detected",
              "Fraction mitochondrial UMIs", "Doublet score")
TESTED = CATEGORICAL + CONTINUOUS

covariates = pd.concat(
    {
        label: covariate_design(
            path, TESTED, scaling=(embeds.get(label).uns.get("covariate_scaling")
                                   if embeds.get(label) is not None else None),
        )
        for label, path in models.items()
    },
    axis=1,
)
covariates.loc[:, (slice(None), ["role", "representation", "reaches_encoder"])]

In [ ]:
# The full picture for one fit, level counts and dispersion coupling included.
LABEL = next(iter(models))
print(LABEL)
covariate_design(models[LABEL], TESTED,
                   scaling=(embeds[LABEL].uns.get("covariate_scaling")
                            if embeds.get(LABEL) is not None else None))

## 3. Training curves

ELBO and reconstruction loss, train against validation, per fit. Every logged loss
component is stored inside `model.pt`, so these need no retraining.

Epoch 0 dominates the y-range, so the panels start at epoch 1. Watch for the validation
curve peeling away from train (overfitting) and for a curve still descending at the last
epoch (undertrained — and note the DDP fits cannot early-stop, so they always use all
`max_epochs`).

**Read the KL-weight panel before comparing any two fits.** Both models anneal the KL term
from 0 upward, and a run whose weight never reaches 1.0 spent its length minimising a
down-weighted KL — which buys a better reconstruction and so a better-looking ELBO. The
reported `elbo = reconstruction + kl_local` is defined identically either way, so the
numbers invite a comparison the schedules do not support.

That is what went wrong in the 200-epoch set: **`scvi+covar` peaked at `kl_weight = 0.4975`**
(scvi-tools' default 400-epoch warmup against a 200-epoch run) while the DRVI fits reached
0.995. Its 34-nat ELBO advantage there is uninterpretable, and was withdrawn. The `_long`
refits resolve it — `S.resolve_kl_warmup` gives both families the same 200-of-600 ramp, so
all three reach 1.0 and spend 400 epochs at a settled objective. **On that matched schedule
scVI is still ahead, by 36.6 nats**, so the finding survives; it was the evidence that had
been invalid, not the result. What did change is scVI's compression: `kl_local` falls from
38.97 to 27.55 and it prunes to 20 of 64 dimensions, against DRVI's 62.

**"Still falling" is a number, not an impression.** The `elbo slope/ep` column is an OLS fit
to the last 50 validation epochs. Every 200-epoch fit sits at its validation minimum on the
final epoch with a slope near −1.4/epoch and no flattening — a power law fitted to those
tails is degenerate, so they say nothing about where the asymptote lies. The bar for
`converged` is a slope shallower than −0.1/epoch; all three `_long` fits clear it
(−0.049, −0.048, −0.096) and none of the earlier fits come close.

In [ ]:
# astype(float): scvi stores history values as objects, which blocks slope arithmetic.
histories = {label: load_history(path).astype(float) for label, path in models.items()}
LOSSES = ["elbo", "reconstruction_loss"]

fig, axes = plt.subplots(
    len(histories), len(LOSSES),
    figsize=(5.2 * len(LOSSES), 3.4 * len(histories)), squeeze=False,
)
for row, (label, hist) in enumerate(histories.items()):
    tail = hist.iloc[1:]
    for col, loss in enumerate(LOSSES):
        ax = axes[row][col]
        for split, style in (("train", "-"), ("validation", "--")):
            key = f"{loss}_{split}"
            if key in tail:
                ax.plot(tail.index, tail[key], style, label=split, lw=1.4)
        ax.set_title(f"{label} — {loss}", fontsize=10)
        ax.set_xlabel("epoch")
        ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

In [ ]:
# Left: the KL annealing schedule. A line stopping short of 1.0 marks a run that never
# optimised the objective its ELBO is reported against, so its losses cannot be compared
# with one that finished the ramp.
# Right: the learning rate. `--reduce-lr-on-plateau` halves it when elbo_validation stalls
# and is the only adaptation available under DDP (early stopping is disabled), so a flat
# line means the fit was still improving steadily rather than that nothing was watching.
fig, (ax_kl, ax_lr) = plt.subplots(1, 2, figsize=(12.4, 3.8))
for label, hist in histories.items():
    if "kl_weight" in hist:
        kw = hist["kl_weight"]
        ax_kl.plot(kw.index, kw, lw=1.6, label=f"{label} (max {kw.max():.3f})")
    if "lr-Adam" in hist:
        lr = hist["lr-Adam"]
        n_drop = int((lr.diff() < 0).sum())
        ax_lr.plot(lr.index, lr, lw=1.6,
                   label=f"{label} ({n_drop} reduction{'s' if n_drop != 1 else ''})")
ax_kl.axhline(1.0, color="k", ls=":", lw=1, zorder=0)
ax_kl.set(xlabel="epoch", ylabel="kl_weight", ylim=(0, 1.08),
          title="KL schedule — short of 1.0 is not comparable")
ax_kl.legend(frameon=False, fontsize=8)
ax_lr.set(xlabel="epoch", ylabel="lr", yscale="log", title="learning rate")
ax_lr.legend(frameon=False, fontsize=8)
if not ax_lr.lines:
    ax_lr.text(0.5, 0.5, "no lr logged\n(--reduce-lr-on-plateau not used)",
               ha="center", va="center", transform=ax_lr.transAxes, fontsize=9)
fig.tight_layout()

In [ ]:
rows = {}
for label, hist in histories.items():
    row = {"epochs": len(hist)}
    for loss in LOSSES:
        tr, va = f"{loss}_train", f"{loss}_validation"
        if tr in hist and va in hist:
            row[f"{loss} train"] = round(hist[tr].iloc[-1], 1)
            row[f"{loss} val"] = round(hist[va].iloc[-1], 1)
            row[f"{loss} val-train"] = round(hist[va].iloc[-1] - hist[tr].iloc[-1], 1)
    if "kl_weight" in hist:
        # below 1.0 => the run optimised a down-weighted KL and its losses are not
        # comparable with a fully warmed-up fit, however good they look
        row["kl_weight max"] = round(hist["kl_weight"].max(), 3)
    if "elbo_validation" in hist:
        val = hist["elbo_validation"].dropna()
        row["best val epoch"] = int(val.idxmin())
        # still improving at the end => the epoch budget, not convergence, stopped it
        row["val still falling"] = bool(val.iloc[-1] <= val.iloc[:-1].min())
        # OLS over the last 50 validation epochs: "still falling" as a rate rather than a
        # yes/no. Shallower than -0.1/epoch is the bar for calling a fit converged.
        n = min(50, len(val))
        row[f"elbo slope/ep (last {n})"] = round(
            float(np.polyfit(np.arange(n), val.to_numpy(float)[-n:], 1)[0]), 3
        )
        row["converged"] = row[f"elbo slope/ep (last {n})"] > -0.1
    rows[label] = row
pd.DataFrame(rows).T

## 4. How much of `n_latent` did each fit use?

DRVI prunes dimensions it does not need and flags them itself (`var["vanished"]`); scVI has
no such notion. The only way to compare the two is one shared rule — a dimension counts as
used when some cell drives it past `|z| >= 0.5` — so `latent_stats` applies that to both and
carries DRVI's own flag alongside where it exists, rather than trusting either blindly.

A collapsed dimension is a VAE working correctly: with more capacity than the data needs,
the KL term pushes the surplus to the prior, where it contributes nothing to either KL or
reconstruction. So a low count is not by itself a defect — read it against the validation
ELBO in section 3. **A model reaching a lower ELBO on fewer dimensions is the more
parsimonious fit, not the weaker one.**

Two things make the count trustworthy rather than an artifact of the 0.5 cutoff: the std
panel below (a bimodal distribution with a clear gap means any cutoff inside the gap gives
the same answer), and the cumulative variance share (how concentrated the representation
is).

**This is why section 6 reports only live dimensions.** Eta-squared is a variance *ratio*,
so it is scale-invariant and cannot tell a live dimension from one sitting at the prior — a
dimension with std 0.008 that wiggles slightly with library scores as high as a real one.
`factor_association` drops collapsed dimensions and logs how many; without that the heatmap
fills with noise that reads as signal.

In [ ]:
usage = {}
for label, embed in embeds.items():
    if embed is None:
        continue
    stats = S.latent_stats(embed)
    var_share = (stats["std"] ** 2).sort_values(ascending=False)
    var_share = var_share / var_share.sum()
    row = {"n_latent": embed.n_vars, "used (|z|>=0.5)": int(stats["used"].sum())}
    if "vanished" in stats:
        row["not vanished (DRVI flag)"] = int((~stats["vanished"]).sum())
        row["rules disagree"] = int((stats["used"] != ~stats["vanished"]).sum())
    row["median std"] = round(float(stats["std"].median()), 3)
    # how concentrated the representation is: 1.0 in the top 25 means the rest are dead
    for n in (5, 10, 25):
        row[f"var share top{n}"] = round(float(var_share.iloc[:n].sum()), 3)
    usage[label] = row
pd.DataFrame(usage).T

In [ ]:
# Sorted per-dimension std, log scale. A bimodal distribution with a clear gap means the
# "used" count is robust to where the threshold sits; a smooth decline means it is not.
fig, ax = plt.subplots(figsize=(7, 4.2))
for label, embed in embeds.items():
    if embed is None:
        continue
    s = S.latent_stats(embed)["std"].sort_values(ascending=False).to_numpy()
    ax.plot(range(1, len(s) + 1), s, marker="o", ms=3, lw=1.2, label=label)
ax.axhline(0.5, color="grey", ls=":", lw=1)
ax.text(1, 0.52, "|z| threshold", color="grey", fontsize=8, va="bottom")
ax.set_yscale("log")
ax.set_xlabel("latent dimension, ranked by std")
ax.set_ylabel("std across cells (log)")
ax.set_title("Is the dimension count robust to the threshold?")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

In [ ]:
for label in DRVI_FITS:
    print(f"--- {label} ---")
    drvi.utils.pl.plot_latent_dimension_stats(embeds[label], ncols=4, remove_vanished=False)

## 5. Latent UMAPs

Identical covariate lists across fits so the panels compare directly. The check that
matters: `Supertype` and `Brain Region` structure should survive, while `library_prep`,
`method` and the QC metrics should *not* organise the embedding — that would mean the
batch correction failed.

In [ ]:
BIOLOGY = ["Supertype", "Brain Region", "Cognitive Status", "CPS_Local"]
TECHNICAL = ["method", "Chemistry", "Number of UMIs", "Fraction mitochondrial UMIs"]

for label, embed in embeds.items():
    if embed is None:
        continue
    print(f"===== {label}: biology =====")
    sc.pl.umap(embed, color=BIOLOGY, ncols=2, wspace=0.3, size=3)

In [ ]:
for label, embed in embeds.items():
    if embed is None:
        continue
    print(f"===== {label}: technical =====")
    sc.pl.umap(embed, color=TECHNICAL, ncols=2, wspace=0.3, size=3)

In [ ]:
# Donor and library are high-cardinality: no legend, look for block structure.
for label, embed in embeds.items():
    if embed is None:
        continue
    print(f"===== {label}: donor / library =====")
    sc.pl.umap(embed, color=["Donor ID", "library_prep"], ncols=2, wspace=0.1,
               size=3, legend_loc=None)

## 6. Latent dimensions against covariates

Eta-squared for categorical covariates (fraction of the dimension's variance explained by
the grouping) and |Spearman rho| for continuous ones, on one 0-1 scale.

**Read eta-squared with the number of levels in mind.** It rises mechanically with group
count, so `library_prep` (902 levels) and `Donor ID` (84) score high almost regardless of
whether a dimension is really a library artifact, while a 2-level covariate scoring high
(e.g. `Sex`) is unambiguous. Compare a dimension's `library_prep` score against its
`Supertype` score (19 levels) rather than against 1.0.

Collapsed dimensions are excluded (see section 4) — `factor_association` prunes them and
logs the count, so a table shorter than `n_latent` is expected, not a missing row.

**Across fits, check the KL weight first.** These scores are variance ratios, so unlike the
ELBO they are not distorted by the loss scale — but the KL weight still shapes them
indirectly, because a weaker penalty compresses the latent less and so leaves *more* of
every covariate readable. The 200-epoch `scvi+covar` fit ran at `kl_weight <= 0.4975`
(section 3), so its associations sit on a less-compressed latent space than the DRVI fits'
and are not a like-for-like comparison. **Use the three `_long` fits for any cross-model
claim** — they share one schedule, all reach `kl_weight = 1.0`, and all converged.

Bear the dimension counts in mind when reading the `_long` rows: scVI concentrates the same
cohort into 20 live dimensions where DRVI spreads it over 62, so a given covariate has far
fewer scVI dimensions to reach its maximum in. A comparable maximum from a third as many
dimensions is the stronger result, not the equal one.

In [ ]:
any_embed = next(e for e in embeds.values() if e is not None)
n_levels = pd.Series(
    {col: any_embed.obs[col].nunique() for col in CATEGORICAL},
    name="n_levels",
).sort_values(ascending=False)
print("categorical levels (eta-squared inflates with these):")
print(n_levels.to_string())

assoc = {
    label: factor_association(embed, categorical=CATEGORICAL, continuous=CONTINUOUS)
    for label, embed in embeds.items()
    if embed is not None
}
{label: a.shape for label, a in assoc.items()}

In [ ]:
fig, axes = plt.subplots(
    1, len(assoc), figsize=(6.5 * len(assoc), 14), squeeze=False,
)
for ax, (label, a) in zip(axes[0], assoc.items(), strict=True):
    sns.heatmap(a, cmap="rocket_r", vmin=0, vmax=1, linewidths=0.2,
                cbar_kws={"label": "eta$^2$ / |Spearman $\\rho$|"}, ax=ax)
    ax.set_title(label)
    ax.set_xlabel("")
    ax.set_ylabel("latent dimension")
fig.tight_layout()

### The headline comparison

Per covariate, the strongest association any dimension reaches, in each fit — beside how
that fit modelled the covariate. Two readings:

- a covariate that is `not modelled` and scores **high** is a latent dimension spent on
  nuisance structure;
- a covariate that *is* modelled and still scores high means the covariate did not
  actually get absorbed — with decoder-only covariates that is the expected failure mode,
  since the encoder never saw it.

In [ ]:
comparison = pd.DataFrame({label: a.max() for label, a in assoc.items()}).round(3)
roles = pd.DataFrame(
    {label: covariates[(label, "role")] for label in assoc}
).reindex(comparison.index)
comparison = pd.concat({"max association": comparison, "role": roles}, axis=1)
comparison.sort_values(("max association", next(iter(assoc))), ascending=False)

In [ ]:
# Strongest association per dimension, as a biological-vs-technical triage.
# Caveat: `library_prep` (902 levels) and `Donor ID` (84) win on level count alone, so a
# dimension is only convincingly technical when it also beats its own Supertype score.
TECHNICAL_COVS = {"library_prep", "Donor ID", "method", "Chemistry", "Number of UMIs",
                  "Genes detected", "Fraction mitochondrial UMIs", "Doublet score"}
for label, a in assoc.items():
    triage = pd.DataFrame({
        "top_covariate": a.idxmax(axis=1),
        "score": a.max(axis=1).round(3),
        "supertype_score": a["Supertype"].round(3),
    })
    triage["kind"] = np.where(triage["top_covariate"].isin(TECHNICAL_COVS),
                              "technical", "biological")
    triage["margin_over_supertype"] = (triage["score"] - triage["supertype_score"]).round(3)
    print(f"===== {label} =====")
    print(triage["kind"].value_counts().to_string())
    display(triage.sort_values("score", ascending=False).head(12))

## 7. Verdict

Fill in after reading the sections above:

- Fits that are sound enough to interpret:
- Dimensions used, per fit (section 4):
- Any overfitting or undertraining (section 3):
- Did DDP change the fit, judged against `drvi 1gpu`?
- Covariates that are still leaking into the latent space despite being modelled (section 6):
- Covariates that are not modelled and should be:

In [ ]:
from session_info2 import session_info

session_info(dependencies=True)